# Venezuela earthquake (2026-06-24) — before/after imagery comparison

Compare **pre-earthquake high-resolution imagery** (Esri Wayback, 2026-05-28 release, sub-meter) against **post-earthquake SkySat/Pelican imagery (50 cm, 26–28 June 2026)** to find evidence of landslides and flag areas for field investigation. A terrain hillshade is one click away for reading slopes and drainages.

- Post-event data: [source.coop/planet/venezuela-earthquake-2026-06-24](https://source.coop/planet/venezuela-earthquake-2026-06-24) — imagery © Planet Labs PBC, **CC BY-NC 4.0**
- Pre-event: [Esri World Imagery Wayback](https://livingatlas.arcgis.com/wayback/) — **capture dates vary by tile**; check the Wayback site before citing a "before" date for a specific slope
- Nothing is downloaded: maps stream tiles from the cloud (needs internet)
- Run cells top to bottom. Change `LOCATION` / `SCENE` and re-run from there to switch areas.

In [1]:
import leafmap

from geer_venezuela import (
    ATTRIBUTION,
    HILLSHADE,
    HILLSHADE_ATTRIBUTION,
    WAYBACK_ATTRIBUTION,
    WAYBACK_PRE_EVENT,
    add_compare_control,
    asset_href,
    load_items,
    locations,
    scenes_for,
)

items = load_items("post-event")
locations(items)

,location,location_slug,scenes,first,last,constellations
0,Caracas,caracas,3,2026-06-26 12:35:41.566000+00:00,2026-06-27 14:58:26.402757+00:00,"pelican, skysat"
1,Catia La Mar,catia-la-mar,1,2026-06-26 11:20:57.920000+00:00,2026-06-26 11:20:57.920000+00:00,skysat
2,Independencia & Ocumare de la Costa,independencia-ocumare,1,2026-06-28 12:20:47.540000+00:00,2026-06-28 12:20:47.540000+00:00,skysat
3,La Guaira,la-guaira,8,2026-06-26 15:05:35.183554+00:00,2026-06-27 11:27:22.914000+00:00,"pelican, skysat"
4,Puerto Cabello,puerto-cabello,2,2026-06-26 11:47:59.786000+00:00,2026-06-26 11:47:59.786000+00:00,skysat
5,Valencia,valencia,1,2026-06-26 19:10:38.440000+00:00,2026-06-26 19:10:38.440000+00:00,skysat
6,Yumare,yumare,1,2026-06-28 11:51:15.058000+00:00,2026-06-28 11:51:15.058000+00:00,skysat


## 1. Overview — where the post-event imagery is

Red outlines are post-event scene footprints over the terrain hillshade. Click a footprint for the scene id and acquisition time.

In [2]:
overview = leafmap.Map(center=(10.5, -67.4), zoom=8)
overview.add_tile_layer(
    HILLSHADE,
    name="Terrain hillshade",
    attribution=HILLSHADE_ATTRIBUTION,
)
footprints = items[
    ["id", "title", "location", "datetime", "constellation", "eo:cloud_cover", "geometry"]
].assign(datetime=lambda d: d["datetime"].astype(str))
overview.add_gdf(
    footprints,
    layer_name="Post-event scene footprints",
    style={"color": "#ff3b30", "weight": 2, "fillOpacity": 0.05},
    zoom_to_layer=False,
)
overview

Map(center=[10.5, -67.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## 2. Pick a location and scene

Set `LOCATION` to one of: `caracas`, `catia-la-mar`, `independencia-ocumare`, `la-guaira`, `puerto-cabello`, `valencia`, `yumare`; `SCENE` is a row number from the table (prefer low `eo:cloud_cover`).

In [3]:
LOCATION = "la-guaira"
SCENE = 0

scenes = scenes_for(items, LOCATION)
scene = scenes.iloc[SCENE]
scenes[["id", "datetime", "constellation", "gsd", "eo:cloud_cover", "title"]]

,id,datetime,constellation,gsd,eo:cloud_cover,title
0,20260626_150535_17_3010,2026-06-26 15:05:35.183554+00:00,pelican,0.62,11,La Guaira — 2026-06-26 15:05 UTC
1,20260626_150536_68_3010,2026-06-26 15:05:36.695705+00:00,pelican,0.62,0,La Guaira — 2026-06-26 15:05 UTC
2,20260626_150538_20_3010,2026-06-26 15:05:38.207857+00:00,pelican,0.62,0,La Guaira — 2026-06-26 15:05 UTC
3,20260627_112621_ssc2_u0002,2026-06-27 11:26:21.284000+00:00,skysat,0.73,15,La Guaira — 2026-06-27 11:26 UTC
4,20260627_112621_ssc2_u0001,2026-06-27 11:26:21.284000+00:00,skysat,0.78,15,La Guaira — 2026-06-27 11:26 UTC
5,20260627_112651_ssc2_u0001,2026-06-27 11:26:51.122000+00:00,skysat,0.66,63,La Guaira — 2026-06-27 11:26 UTC
6,20260627_112651_ssc2_u0002,2026-06-27 11:26:51.122000+00:00,skysat,0.67,63,La Guaira — 2026-06-27 11:26 UTC
7,20260627_112722_ssc2_u0001,2026-06-27 11:27:22.914000+00:00,skysat,0.79,58,La Guaira — 2026-06-27 11:27 UTC


## 3. Compare — BEFORE / AFTER / TOPO

One map, three views, switched by the **button bar pinned to the top-left**:

- **BEFORE** — Esri Wayback 2026-05-28 (sub-meter)
- **AFTER** — post-event 50 cm scene
- **TOPO** — terrain hillshade, for reading slopes, drainages, and runout paths

Click BEFORE/AFTER repeatedly to flicker — new landslide scars pop out. Flip to TOPO to see whether a scar sits on a steep face or above a channel.

In [17]:
m = leafmap.Map()
m.add_tile_layer(
    HILLSHADE,
    name="TOPO — terrain hillshade",
    attribution=HILLSHADE_ATTRIBUTION,
)
m.add_tile_layer(
    WAYBACK_PRE_EVENT,
    name="BEFORE — Esri Wayback 2026-05-28",
    attribution=WAYBACK_ATTRIBUTION,
    max_zoom=19,
)
m.add_cog_layer(
    asset_href(scene, "visual"),
    name=f"AFTER — {scene['title']}",
    attribution=ATTRIBUTION,
)
add_compare_control(
    m,
    {
        "BEFORE": "BEFORE — Esri Wayback 2026-05-28",
        "AFTER": f"AFTER — {scene['title']}",
        "TOPO": "TOPO — terrain hillshade",
    },
    selected="AFTER",
)
m

Map(center=[-66.98433806397188, 10.540821876664467], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## 4. Mark landslide candidates for field teams

Use the **draw tools** (left edge of the map) to drop points or polygons on suspected landslides — in any view — then run the next cell to export them as GeoJSON (loads directly in QGIS, ArcGIS Online, Google Earth, etc.).

In [29]:
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "landslide_candidates"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"{LOCATION}_{scene['id']}.geojson"

if m.user_rois is not None and m.user_rois["features"]:
    m.save_draw_features(str(out_file))
    print(f"Saved {len(m.user_rois['features'])} feature(s) to {out_file}")
else:
    print("Nothing drawn yet — use the draw tools on the map above, then re-run this cell.")

Saved 44 feature(s) to /Users/lornearnold/GitHub/GEER_Venezuela/data/landslide_candidates/la-guaira_20260626_150535_17_3010.geojson


## What to look for

- **Fresh scars**: light-toned (tan/grey) patches of bare soil/rock on vegetated slopes that are absent in the BEFORE image.
- **Debris runouts**: fans or flow paths below scars — down drainages, across roads, into buildings (trace the channel in TOPO view).
- **Blocked drainages / turbid water**: sediment plumes at river mouths, ponding upstream of debris.
- **Road cuts and coastal bluffs**: common failure points; scan along the Caracas–La Guaira corridor.

Caveats:

- Wayback capture dates vary by tile and can predate the release by months–years — verify at the [Wayback site](https://livingatlas.arcgis.com/wayback/) before citing a "before" date.
- Check `eo:cloud_cover` and prefer clear scenes; the `udm2` asset of each scene is a per-pixel cloud/shadow mask if needed.

## Ideas for additional data (later)

- **Sentinel-2** (10 m, free, ~5-day revisit) via Earth Search STAC — regional sweep beyond the Planet footprints.
- **Maxar Open Data Program** — often releases 30–50 cm imagery for major disasters (not activated for this event as of 2026-07-03).
- **Copernicus EMS / UNOSAT** rapid-mapping activations — may already have damage/landslide vectors.
- **NASA/USGS**: ShakeMap + slope data to prioritize where landslides are *likely*, not just visible.